# 00b — Synthetic Nepali Financial Document Generation

Generates synthetic Nepali financial document PDFs using ReportLab.
Covers document types common in Nepal:

| Type | Description |
|---|---|
| Balance Sheet | Assets / Liabilities / Equity |
| Profit & Loss | Income statement |
| Cash Flow | Operating / Investing / Financing |
| Bank Statement | Transaction ledger |
| Loan Schedule | EMI amortisation table |

Devanagari text (Nepali headers) is included via a bundled Noto Sans Devanagari font.

Output: PDFs in `data/pdfs/synthetic/`, then extracted to `data/raw/` by notebook 00a.

In [ ]:
# pip install reportlab faker
import sys
sys.path.insert(0, '..')

import random
import urllib.request
from pathlib import Path
from datetime import date, timedelta
from decimal import Decimal

from faker import Faker
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Table, TableStyle, Paragraph,
    Spacer, HRFlowable,
)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

SYNTH_PDF_DIR = Path('../data/pdfs/synthetic')
SYNTH_PDF_DIR.mkdir(parents=True, exist_ok=True)

fake = Faker()
random.seed(42)

In [ ]:
# Download Noto Sans Devanagari for Nepali text rendering
FONT_PATH = Path('../data/NotoSansDevanagari-Regular.ttf')
FONT_URL = 'https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Regular.ttf'

if not FONT_PATH.exists():
    print('Downloading Noto Sans Devanagari font...')
    urllib.request.urlretrieve(FONT_URL, FONT_PATH)
    print(f'Saved to {FONT_PATH}')
else:
    print(f'Font already present: {FONT_PATH}')

pdfmetrics.registerFont(TTFont('Devanagari', str(FONT_PATH)))

In [ ]:
# --- Nepali company + bank name pools ---
NEPALI_BANKS = [
    'Nabil Bank Limited', 'NIC Asia Bank Ltd', 'Everest Bank Limited',
    'Nepal Bank Limited', 'Rastriya Banijya Bank', 'Himalayan Bank Ltd',
    'Global IME Bank Ltd', 'Prabhu Bank Limited', 'Kumari Bank Limited',
    'Sanima Bank Limited', 'Machhapuchchhre Bank Ltd', 'Laxmi Sunrise Bank',
]
NEPALI_COMPANIES = [
    'Nepal Telecom', 'Nepal Oil Corporation', 'Nepal Airlines Corporation',
    'Chilime Hydropower Company', 'Upper Tamakoshi Hydropower',
    'Bottlers Nepal Limited', 'Surya Nepal Pvt Ltd', 'Unilever Nepal Ltd',
    'Siddhartha Insurance Ltd', 'Sagarmatha Insurance',
]
FISCAL_YEARS = ['2078/79', '2079/80', '2080/81', '2081/82']

# Nepali labels (Devanagari) for headers
NEP_LABELS = {
    'balance_sheet':   'स्थिति विवरण',
    'profit_loss':     'नाफा-नोक्सान हिसाब',
    'cash_flow':       'नगद प्रवाह विवरण',
    'bank_statement':  'बैंक स्टेटमेन्ट',
    'loan_schedule':   'ऋण तालिका',
    'amount':          'रकम (रू.)',
    'particulars':     'विवरण',
    'fiscal_year':     'आर्थिक वर्ष',
    'date':            'मिति',
    'balance':         'मौज्दात',
}

def nepali_amount():
    """Random NPR amount formatted with comma grouping (lakh/crore style)."""
    val = random.randint(10_000, 50_000_000)
    # Nepal uses lakh grouping: 1,00,000
    s = str(val)
    if len(s) > 3:
        s = s[:-3] + ',' + s[-3:]
    if len(s) > 6:
        s = s[:-6] + ',' + s[-6:]
    return s

def bs_date():
    """Random Bikram Sambat date string."""
    year = random.randint(2078, 2081)
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    return f'{year}-{month:02d}-{day:02d} (B.S.)'

print('Helpers ready.')

In [ ]:
def base_styles():
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        'NepalTitle',
        parent=styles['Title'],
        fontName='Devanagari',
        fontSize=14,
        spaceAfter=4,
    )
    sub_style = ParagraphStyle(
        'NepalSub',
        parent=styles['Normal'],
        fontName='Devanagari',
        fontSize=10,
        spaceAfter=2,
    )
    return styles, title_style, sub_style


TABLE_STYLE = TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1B3A6B')),
    ('TEXTCOLOR',  (0, 0), (-1, 0), colors.white),
    ('FONTNAME',   (0, 0), (-1, 0), 'Devanagari'),
    ('FONTSIZE',   (0, 0), (-1, 0), 9),
    ('FONTNAME',   (0, 1), (-1, -1), 'Helvetica'),
    ('FONTSIZE',   (0, 1), (-1, -1), 8),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#F0F4FA')]),
    ('GRID',       (0, 0), (-1, -1), 0.4, colors.HexColor('#CCCCCC')),
    ('ALIGN',      (1, 0), (-1, -1), 'RIGHT'),
    ('TOPPADDING', (0, 0), (-1, -1), 3),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
])

print('Styles defined.')

In [ ]:
def generate_balance_sheet(company: str, fy: str, path: Path):
    doc = SimpleDocTemplate(str(path), pagesize=A4,
                            topMargin=1.5*cm, bottomMargin=1.5*cm,
                            leftMargin=2*cm, rightMargin=2*cm)
    styles, title_style, sub_style = base_styles()
    story = [
        Paragraph(company, title_style),
        Paragraph(f'{NEP_LABELS["balance_sheet"]} | Balance Sheet', title_style),
        Paragraph(f'{NEP_LABELS["fiscal_year"]}: {fy}  |  As at {bs_date()}', sub_style),
        HRFlowable(width='100%', thickness=1, color=colors.HexColor('#1B3A6B')),
        Spacer(1, 0.3*cm),
    ]

    assets = [
        [NEP_LABELS['particulars'] + ' / Particulars', NEP_LABELS['amount'] + ' (Current)', NEP_LABELS['amount'] + ' (Previous)'],
        ['ASSETS', '', ''],
        ['Cash and Cash Equivalents', nepali_amount(), nepali_amount()],
        ['Loans and Advances', nepali_amount(), nepali_amount()],
        ['Investment Securities', nepali_amount(), nepali_amount()],
        ['Fixed Assets (Net)', nepali_amount(), nepali_amount()],
        ['Other Assets', nepali_amount(), nepali_amount()],
        ['Total Assets', nepali_amount(), nepali_amount()],
        ['', '', ''],
        ['LIABILITIES & EQUITY', '', ''],
        ['Deposits from Customers', nepali_amount(), nepali_amount()],
        ['Borrowings', nepali_amount(), nepali_amount()],
        ['Other Liabilities', nepali_amount(), nepali_amount()],
        ['Share Capital', nepali_amount(), nepali_amount()],
        ['Retained Earnings', nepali_amount(), nepali_amount()],
        ['Total Liabilities & Equity', nepali_amount(), nepali_amount()],
    ]
    t = Table(assets, colWidths=[9*cm, 4*cm, 4*cm])
    t.setStyle(TABLE_STYLE)
    story.append(t)
    doc.build(story)


def generate_profit_loss(company: str, fy: str, path: Path):
    doc = SimpleDocTemplate(str(path), pagesize=A4,
                            topMargin=1.5*cm, bottomMargin=1.5*cm,
                            leftMargin=2*cm, rightMargin=2*cm)
    styles, title_style, sub_style = base_styles()
    story = [
        Paragraph(company, title_style),
        Paragraph(f'{NEP_LABELS["profit_loss"]} | Profit & Loss Account', title_style),
        Paragraph(f'{NEP_LABELS["fiscal_year"]}: {fy}', sub_style),
        HRFlowable(width='100%', thickness=1, color=colors.HexColor('#1B3A6B')),
        Spacer(1, 0.3*cm),
    ]

    rows = [
        [NEP_LABELS['particulars'] + ' / Particulars', 'Current Year', 'Previous Year'],
        ['Interest Income',         nepali_amount(), nepali_amount()],
        ['Commission Income',       nepali_amount(), nepali_amount()],
        ['Foreign Exchange Income', nepali_amount(), nepali_amount()],
        ['Other Operating Income',  nepali_amount(), nepali_amount()],
        ['Total Income',            nepali_amount(), nepali_amount()],
        ['', '', ''],
        ['Interest Expense',        nepali_amount(), nepali_amount()],
        ['Staff Expenses',          nepali_amount(), nepali_amount()],
        ['Operating Expenses',      nepali_amount(), nepali_amount()],
        ['Loan Loss Provisions',    nepali_amount(), nepali_amount()],
        ['Depreciation',            nepali_amount(), nepali_amount()],
        ['Total Expenses',          nepali_amount(), nepali_amount()],
        ['', '', ''],
        ['Profit Before Tax',       nepali_amount(), nepali_amount()],
        ['Income Tax Expense',      nepali_amount(), nepali_amount()],
        ['Net Profit / (Loss)',     nepali_amount(), nepali_amount()],
    ]
    t = Table(rows, colWidths=[9*cm, 4*cm, 4*cm])
    t.setStyle(TABLE_STYLE)
    story.append(t)
    doc.build(story)


def generate_bank_statement(bank: str, path: Path, n_rows: int = 30):
    doc = SimpleDocTemplate(str(path), pagesize=A4,
                            topMargin=1.5*cm, bottomMargin=1.5*cm,
                            leftMargin=1.5*cm, rightMargin=1.5*cm)
    styles, title_style, sub_style = base_styles()

    account = f'{random.randint(100,999)}-{random.randint(10,99)}-{random.randint(1000000,9999999)}'
    story = [
        Paragraph(bank, title_style),
        Paragraph(NEP_LABELS['bank_statement'] + ' | Account Statement', title_style),
        Paragraph(f'Account No: {account}  |  Period: {bs_date()} to {bs_date()}', sub_style),
        HRFlowable(width='100%', thickness=1, color=colors.HexColor('#1B3A6B')),
        Spacer(1, 0.3*cm),
    ]

    header = [NEP_LABELS['date'] + ' / Date', 'Particulars', 'Debit (Rs.)', 'Credit (Rs.)', NEP_LABELS['balance'] + ' / Balance']
    rows = [header]
    balance = random.randint(10000, 500000)
    for _ in range(n_rows):
        txn_type = random.choice(['DR', 'CR'])
        amount = random.randint(100, 100000)
        narration = random.choice([
            'ATM Withdrawal', 'NEFT Transfer', 'Cheque Deposit', 'Mobile Banking',
            'Salary Credit', 'Loan EMI', 'Utility Payment', 'Internet Banking',
            'Fund Transfer', 'Cash Deposit',
        ])
        if txn_type == 'DR':
            balance -= amount
            rows.append([bs_date(), narration, f'{amount:,}', '', f'{balance:,}'])
        else:
            balance += amount
            rows.append([bs_date(), narration, '', f'{amount:,}', f'{balance:,}'])

    t = Table(rows, colWidths=[3*cm, 6*cm, 3*cm, 3*cm, 3*cm])
    t.setStyle(TABLE_STYLE)
    story.append(t)
    doc.build(story)


def generate_loan_schedule(bank: str, path: Path, months: int = 36):
    doc = SimpleDocTemplate(str(path), pagesize=A4,
                            topMargin=1.5*cm, bottomMargin=1.5*cm,
                            leftMargin=1.5*cm, rightMargin=1.5*cm)
    styles, title_style, sub_style = base_styles()

    principal = random.randint(500000, 5000000)
    rate = random.uniform(10.0, 15.0)
    story = [
        Paragraph(bank, title_style),
        Paragraph(NEP_LABELS['loan_schedule'] + ' | Loan Amortisation Schedule', title_style),
        Paragraph(f'Principal: Rs. {principal:,}  |  Rate: {rate:.2f}%  |  Tenure: {months} months', sub_style),
        HRFlowable(width='100%', thickness=1, color=colors.HexColor('#1B3A6B')),
        Spacer(1, 0.3*cm),
    ]

    monthly_rate = rate / 100 / 12
    emi = principal * monthly_rate * (1 + monthly_rate)**months / ((1 + monthly_rate)**months - 1)

    header = ['Month', 'EMI (Rs.)', 'Principal (Rs.)', 'Interest (Rs.)', 'Balance (Rs.)']
    rows = [header]
    bal = float(principal)
    for m in range(1, months + 1):
        interest = bal * monthly_rate
        princ = emi - interest
        bal -= princ
        rows.append([str(m), f'{emi:,.0f}', f'{princ:,.0f}', f'{interest:,.0f}', f'{max(bal,0):,.0f}'])

    t = Table(rows, colWidths=[2*cm, 3.5*cm, 3.5*cm, 3.5*cm, 3.5*cm])
    t.setStyle(TABLE_STYLE)
    story.append(t)
    doc.build(story)

print('Document generators defined.')

In [ ]:
N_PER_TYPE = 20  # number of synthetic PDFs per document type

generated = []

for i in range(N_PER_TYPE):
    company = random.choice(NEPALI_COMPANIES + NEPALI_BANKS)
    bank    = random.choice(NEPALI_BANKS)
    fy      = random.choice(FISCAL_YEARS)

    bs_path  = SYNTH_PDF_DIR / f'balance_sheet_{i:03d}.pdf'
    pl_path  = SYNTH_PDF_DIR / f'profit_loss_{i:03d}.pdf'
    sta_path = SYNTH_PDF_DIR / f'bank_statement_{i:03d}.pdf'
    loan_path = SYNTH_PDF_DIR / f'loan_schedule_{i:03d}.pdf'

    generate_balance_sheet(company, fy, bs_path)
    generate_profit_loss(company, fy, pl_path)
    generate_bank_statement(bank, sta_path)
    generate_loan_schedule(bank, loan_path)

    generated += [bs_path, pl_path, sta_path, loan_path]

print(f'Generated {len(generated)} synthetic PDFs in {SYNTH_PDF_DIR}')

In [ ]:
# Convert synthetic PDFs to images (same as real PDFs)
import fitz
from pathlib import Path

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(exist_ok=True)

synth_images = []
for pdf_path in sorted(SYNTH_PDF_DIR.glob('*.pdf')):
    doc = fitz.open(str(pdf_path))
    mat = fitz.Matrix(150/72, 150/72)
    for i in range(len(doc)):
        pix = doc[i].get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        img_path = RAW_DIR / f'{pdf_path.stem}_page{i+1:04d}.png'
        pix.save(str(img_path))
        synth_images.append(img_path)
    doc.close()

print(f'Extracted {len(synth_images)} page images from synthetic PDFs')

In [ ]:
# Preview generated documents
import matplotlib.pyplot as plt
import cv2
import random

samples = random.sample(synth_images, min(8, len(synth_images)))
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for ax, path in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(path.stem[:35], fontsize=7)
    ax.axis('off')
plt.suptitle('Synthetic Nepali Financial Documents', fontsize=13)
plt.tight_layout()
plt.show()